[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FieldQuantum/fieldqkit/blob/main/examples/demo_noisy_simulation.ipynb)

In [ ]:
# 在 Google Colab 上运行请先执行本单元格安装依赖（本地已安装可跳过）
%pip install -q "fieldqkit[sim]"

# 含噪声线路：构建、密度矩阵模拟与误差缓解

本教程演示 `fieldqkit` 的噪声信道支持：

1. **构建含噪线路** —— 7 种噪声信道（去极化、比特/相位翻转、振幅/相位阻尼）；
2. **密度矩阵模拟** —— 含噪线路自动路由到密度矩阵后端，跟踪相干性衰减；
3. **采样与期望值** —— `simulate_counts` / `expectation_pauli` 自动识别含噪线路；
4. **退相干扫描** —— 噪声强度对关联与相干性的影响；
5. **误差缓解** —— 与 CDR（Clifford 数据回归）。

> 约定：噪声率必须是**具体数字**（不支持符号化/可微参数）。含噪线路只能在模拟器上运行（本地 `Simulator` 或云端 `fieldquantum_sim`），提交到真机会被拒绝，并且**跳过硬件编译**。

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from fieldqkit import QuantumHardwareClient, QuantumCircuit
from fieldqkit.sim import simulate_density_matrix, simulate_counts, expectation_pauli
from fieldqkit.circuit.quantumcircuit_helpers import has_noise_channels


def section(title: str):
    print("\n" + "=" * 18, title, "=" * 18)

## 1) 构建含噪线路

所有噪声信道都是链式 API，和普通门一样直接追加：

| 方法 | 含义 | 参数 |
|---|---|---|
| `depolarize1(p, q)` | 单比特去极化 | 概率 `p` |
| `depolarize2(p, q0, q1)` | 两比特去极化 | 概率 `p` |
| `x_error/y_error/z_error(p, q)` | 单比特 Pauli 翻转 | 概率 `p` |
| `amplitude_damping(γ, q)` | 振幅阻尼（T1 弛豫） | 阻尼率 `γ` |
| `phase_damping(γ, q)` | 相位阻尼（T2 退相干） | 阻尼率 `γ` |

In [ ]:
qc = QuantumCircuit(2)
qc.h(0).cx(0, 1)
qc.depolarize1(0.05, 0)       # 单比特去极化
qc.depolarize2(0.03, 0, 1)    # 两比特去极化
qc.amplitude_damping(0.10, 0) # T1 弛豫
qc.phase_damping(0.10, 1)     # T2 退相干
qc.x_error(0.02, 0)           # 比特翻转
qc.z_error(0.02, 1)           # 相位翻转

print("contains noise channels:", has_noise_channels(qc))
print("gates:")
for g in qc.gates:
    print("  ", g)

In [ ]:
# 噪声门也能序列化到 OpenQASM 2.0（以 opaque 声明导出）
print(qc.to_openqasm2())

In [ ]:
# 噪声率必须是具体数字；传入符号会被拒绝
try:
    QuantumCircuit(1).depolarize1("p", 0)
except ValueError as e:
    print("rejected symbolic rate:", e)

# remove_noise_channels() 返回去掉噪声门的理想线路副本（用作 noise-free 参考）
ideal_copy = qc.remove_noise_channels()
print("ideal copy gates:", [g[0] for g in ideal_copy.gates])

## 2) 密度矩阵模拟

含噪线路无法用纯态描述，`simulate_density_matrix` 返回 $2^n\times 2^n$ 密度矩阵 $\rho$。无噪声的纯态对应秩 1 的 $\rho=|\psi\rangle\langle\psi|$；噪声让 $\rho$ 变成混合态，并衰减非对角的相干项。

> 在 GPU 环境下模拟器返回的张量在 CUDA 上，下面统一用 `.cpu()` 取回再展示。

In [ ]:
# 干净的 Bell 态：rho = |Phi+><Phi+|
bell = QuantumCircuit(2)
bell.h(0).cx(0, 1)
rho_clean = simulate_density_matrix(bell).cpu()

print("trace(rho) =", round(float(torch.trace(rho_clean).real), 6))
print("rho (clean Bell):")
print(np.round(rho_clean.numpy(), 3))

In [ ]:
# 加去极化噪声后：对角占据不变，非对角相干项被压低 -> 混合态
bell_noisy = QuantumCircuit(2)
bell_noisy.h(0).cx(0, 1)
bell_noisy.depolarize1(0.20, 0).depolarize1(0.20, 1)
rho_noisy = simulate_density_matrix(bell_noisy).cpu()

print("trace(rho) =", round(float(torch.trace(rho_noisy).real), 6))
print("rho (noisy Bell):")
print(np.round(rho_noisy.numpy(), 3))

zz_clean = float(expectation_pauli(rho_clean, "ZZ", num_qubits=2))
zz_noisy = float(expectation_pauli(rho_noisy, "ZZ", num_qubits=2))
print(f"\n<ZZ> clean = {zz_clean:+.4f}   <ZZ> noisy = {zz_noisy:+.4f}")

## 3) 采样：`simulate_counts` 自动路由

`simulate_counts` 会检测线路是否含噪：含噪则走密度矩阵后端，从 $\rho$ 的对角线（测量概率）采样；否则走 statevector/MPS。无需手动切换。

In [ ]:
clean_m = QuantumCircuit(2)
clean_m.h(0).cx(0, 1).measure_all()

noisy_m = QuantumCircuit(2)
noisy_m.h(0).cx(0, 1).depolarize1(0.15, 0).depolarize1(0.15, 1).measure_all()

print("clean counts:", simulate_counts(clean_m, shots=2000, seed=0))
print("noisy counts:", simulate_counts(noisy_m, shots=2000, seed=0))

## 4) 退相干扫描

扫描噪声强度，观察其对**关联**（去极化下的 $\langle ZZ\rangle$）和**相干性**（相位阻尼下的 $|\rho_{01}|$、振幅阻尼下的激发态占据 $\rho_{11}$）的影响。

In [ ]:
ps = np.linspace(0.0, 0.5, 11)
zz_vs_p = []
for p in ps:
    c = QuantumCircuit(2)
    c.h(0).cx(0, 1)
    if p > 0:
        c.depolarize1(float(p), 0).depolarize1(float(p), 1)
    r = simulate_density_matrix(c).cpu()
    zz_vs_p.append(float(expectation_pauli(r, "ZZ", num_qubits=2)))

gammas = np.linspace(0.0, 1.0, 11)
coh_phase = []   # |rho_01| under phase damping on |+>
pop_amp = []     # rho_11 under amplitude damping on |1>
for g in gammas:
    c = QuantumCircuit(1); c.h(0)
    if g > 0:
        c.phase_damping(float(g), 0)
    coh_phase.append(float(simulate_density_matrix(c).cpu()[0, 1].abs()))

    c = QuantumCircuit(1); c.x(0)
    if g > 0:
        c.amplitude_damping(float(g), 0)
    pop_amp.append(float(simulate_density_matrix(c).cpu()[1, 1].real))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ps, zz_vs_p, "o-")
axes[0].set_xlabel("depolarizing p"); axes[0].set_ylabel(r"$\langle ZZ\rangle$")
axes[0].set_title("Bell correlation vs depolarizing noise"); axes[0].grid(alpha=0.3)
axes[1].plot(gammas, coh_phase, "o-", label=r"phase damping: $|\rho_{01}|$")
axes[1].plot(gammas, pop_amp, "s-", label=r"amplitude damping: $\rho_{11}$")
axes[1].set_xlabel(r"damping $\gamma$"); axes[1].set_ylabel("value")
axes[1].set_title("Coherence / population decay"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5) 误差缓解

### 5b) CDR（Clifford 数据回归 / Clifford fitting）

通过 `client.run_auto(..., clifford_fitting=True)` 一站式完成。框架会：

1. 在含噪后端跑主任务，得到 raw 期望；
2. 生成若干 Clifford-随机化的校准线路；
3. 每条校准线路：含噪分支在含噪模拟器上测量，**理想分支自动去掉噪声门**后用 Clifford(+T) 模拟器精确求值；
4. 对每个观测量拟合 $E_\text{ideal}\approx a\,E_\text{noisy}+b$ 并应用到主任务结果。

这里后端用本地 `Simulator`，线路里带显式噪声信道，CDR 负责把它缓解掉。

In [ ]:
n = 3
base = QuantumCircuit(n)
for q in range(n):
    base.h(q)
for q in range(n - 1):
    base.cx(q, q + 1)
for q in range(n):
    base.ry(0.4 + 0.2 * q, q)

# 加显式噪声信道
noisy = base.deepcopy()
for q in range(n):
    noisy.depolarize1(0.08, q)

observables = ["ZII", "IZI", "ZZI", "XIX"]
client = QuantumHardwareClient()

In [ ]:
section("ideal reference (noise stripped)")
ideal = client.run_auto(
    circuit=base.remove_noise_channels(), name="noise_ideal", num_qubits=n,
    shots=8192, observables=observables, prefer_chips="Simulator", print_true=False,
)

section("raw (noisy, no mitigation)")
raw = client.run_auto(
    circuit=noisy, name="noise_raw", num_qubits=n,
    shots=8192, observables=observables, prefer_chips="Simulator", print_true=False,
)

section("noisy + clifford fitting (CDR)")
cdr = client.run_auto(
    circuit=noisy, name="noise_cdr", num_qubits=n,
    shots=8192, observables=observables, prefer_chips="Simulator",
    clifford_fitting=True, clifford_fitting_num_samples=8,
    clifford_fitting_num_non_clifford_gates=2, clifford_fitting_seed=7,
    print_true=False,
)

iv, rv, cv = ideal.observable_values, raw.observable_values, cdr.observable_values
print(f"\n{'obs':6}{'ideal':>9}{'noisy':>9}{'cdr':>9}")
for o in observables:
    print(f"{o:6}{iv[o]:+9.4f}{rv[o]:+9.4f}{cv[o]:+9.4f}")

l1 = lambda d: sum(abs(d[o] - iv[o]) for o in observables) / len(observables)
print(f"\nmean |E - E_ideal|:  noisy={l1(rv):.4f}   cdr={l1(cv):.4f}")

In [ ]:
xs = np.arange(len(observables))
w = 0.25
plt.figure(figsize=(8, 4.5))
plt.bar(xs - w, [rv[o] for o in observables], w, label="noisy (raw)")
plt.bar(xs,     [cv[o] for o in observables], w, label="+CDR")
plt.bar(xs + w, [iv[o] for o in observables], w, label="ideal")
plt.axhline(0, color="black", lw=0.5)
plt.xticks(xs, observables)
plt.ylabel(r"$\langle P\rangle$")
plt.title("Clifford data regression on a noisy circuit (local Simulator)")
plt.legend(); plt.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

## 6) 真机保护

含噪线路无法在真机上执行（噪声信道没有硬件基门分解），提交到真机后端会被直接拒绝。

In [ ]:
from fieldqkit.api.backend import is_noisy_circuit_for_backend

try:
    is_noisy_circuit_for_backend(noisy, "Baihua")
except ValueError as e:
    print("rejected on hardware:", e)

print("allowed on Simulator:", is_noisy_circuit_for_backend(noisy, "Simulator"))
print("allowed on fieldquantum_sim:", is_noisy_circuit_for_backend(noisy, "fieldquantum_sim"))

## 小结

- 7 种噪声信道以链式 API 构建，噪声率为具体数字；
- `simulate_density_matrix` / `simulate_counts` / `expectation_pauli` 自动识别并走密度矩阵后端；
- 误差缓解：**CDR**（`clifford_fitting=True`，理想分支自动剥离噪声门）；本例中 CDR 把平均误差降低了约一个数量级；
- 含噪线路仅限模拟器（本地 `Simulator` / 云端 `fieldquantum_sim`），真机提交会被拒绝且跳过硬件编译。